# Markov Decision Processes: Policies, Value Iteration, and Policy Iteration
This module adds decision-making to the Markov models of the earlier modules. A Markov model describes how a system moves between states on its own; a __Markov decision process (MDP)__ lets an agent choose actions that influence those transitions and earn rewards. The agent's task is to find a policy, a rule for choosing actions, that maximizes its expected long-run reward. This is the same maximum-expected-utility objective from Module 1, now applied to a sequence of decisions over time.

> __Learning Objectives.__
>
> By the end of this module, you will be able to define and demonstrate mastery of the following key concepts:

> * __Formulate a Markov decision process:__ Represent a sequential decision problem as the tuple $(\mathcal{S},\mathcal{A},P,R,\gamma)$ of states, actions, transition probabilities, rewards, and a discount factor, and state the agent's objective as maximizing the expected discounted return.
> * __Policies and value functions:__ Define a policy $\pi$, the state-value function $V^{\pi}(s)$, and the action-value function $Q^{\pi}(s,a)$, and write the Bellman expectation and Bellman optimality equations that they satisfy.
> * __Solve for an optimal policy:__ Compute the optimal value function and policy with value iteration and with policy iteration, state the convergence guarantees for each (contraction mapping and the policy improvement theorem), and compare the two methods.

By the end you will be able to model a sequential decision problem and compute the policy that solves it. Let's get started!
___


## Markov Decision Processes
A Markov decision process models decision-making when outcomes are partly random and partly controlled by an agent. An MDP is the tuple $\left(\mathcal{S}, \mathcal{A}, P, R, \gamma\right)$:

* __States__ $\mathcal{S}$: the set of all states $s\in\mathcal{S}$ the system can occupy, as in a Markov model. Example: investor moods $\mathcal{S} \equiv \{\text{bullish},\text{neutral},\text{bearish}\}$.
* __Actions__ $\mathcal{A}$: the set of all actions $a\in\mathcal{A}$ available to the agent, where $\mathcal{A}_{s}\subseteq\mathcal{A}$ is the subset accessible from state $s$. Example: $\mathcal{A} \equiv \{\text{buy},\text{hold},\text{sell}\}$.
* __Transitions__ $P$: the transition model $P(s^{\prime}\mid s,a) = \Pr(s_{t+1}=s^{\prime}\mid s_t=s, a_t=a)$ is the probability that action $a$ in state $s$ at time $t$ leads to state $s^{\prime}$ at time $t+1$. This has the Markov property, but now conditions on both the current state $s$ and the action $a$.
* __Reward__ $R$: $R(s,a)$ is the expected immediate reward for taking action $a$ in state $s$. When the reward depends on the next state, $R(s,a)=\sum_{s^{\prime}\in\mathcal{S}}P(s^{\prime}\mid s,a)\,R(s,a,s^{\prime})$.
* __Discount__ $\gamma$: the discount factor $0\le\gamma<1$ weighs future rewards relative to immediate rewards. A reward received $k$ steps in the future is worth $\gamma^{k}$ times its face value.

### Policy and return
A __policy__ $\pi:\mathcal{S}\rightarrow\mathcal{A}$ maps each state to an action (a deterministic policy); more generally a stochastic policy $\pi(a\mid s)$ gives a probability distribution over actions. Following a policy from time $t$ produces a stream of rewards, summarized by the __discounted return__:
$$
G_t = \sum_{k=0}^{\infty} \gamma^{k}\, R_{t+k+1},
$$
where $R_{t+k+1}$ is the reward received at step $t+k+1$. The discount factor $\gamma<1$ keeps this sum finite and makes the agent prefer rewards sooner. The agent's goal is to find a policy that maximizes the expected discounted return $\mathbb{E}_{\pi}[G_t]$ from every state. The next section makes that objective precise with value functions.
___


## Policies and Value Functions
To compare policies we measure how good each state (and each state-action pair) is under a policy.

### State- and action-value functions
The __state-value function__ $V^{\pi}:\mathcal{S}\rightarrow\mathbb{R}$ is the expected discounted return starting from state $s$ and following policy $\pi$:
$$
V^{\pi}(s) = \mathbb{E}_{\pi}\!\left[\,G_t \mid s_t = s\,\right].
$$
The __action-value function__ $Q^{\pi}:\mathcal{S}\times\mathcal{A}\rightarrow\mathbb{R}$ is the expected discounted return starting from state $s$, taking action $a$, and following $\pi$ thereafter:
$$
Q^{\pi}(s,a) = \mathbb{E}_{\pi}\!\left[\,G_t \mid s_t = s,\, a_t = a\,\right].
$$
For a deterministic policy the two are related by $V^{\pi}(s) = Q^{\pi}(s,\pi(s))$.

### The Bellman expectation equation
Splitting the return into the immediate reward plus the discounted return from the next state gives the __Bellman expectation equation__. For a deterministic policy $\pi$:
$$
V^{\pi}(s) = \underbrace{R(s,\pi(s))}_{\text{immediate}} + \gamma\underbrace{\sum_{s^{\prime}\in\mathcal{S}} P(s^{\prime}\mid s,\pi(s))\,V^{\pi}(s^{\prime})}_{\text{expected future value}},
\qquad
Q^{\pi}(s,a) = R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}} P(s^{\prime}\mid s,a)\,V^{\pi}(s^{\prime}).
$$
This is a system of $|\mathcal{S}|$ linear equations in the unknowns $\{V^{\pi}(s)\}$, one per state. We use it directly in policy evaluation.

### Optimal value functions and the Bellman optimality equation
A policy $\pi^{\star}$ is __optimal__ if its value is at least as large as that of any other policy in every state. The optimal value functions are
$$
V^{\star}(s) = \max_{\pi} V^{\pi}(s),
\qquad
Q^{\star}(s,a) = \max_{\pi} Q^{\pi}(s,a).
$$
They satisfy the __Bellman optimality equation__, which replaces the policy's action with a maximization over actions:
$$
V^{\star}(s) = \max_{a\in\mathcal{A}_{s}}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}} P(s^{\prime}\mid s,a)\,V^{\star}(s^{\prime})\right),
\qquad
Q^{\star}(s,a) = R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}} P(s^{\prime}\mid s,a)\,V^{\star}(s^{\prime}).
$$
Given $Q^{\star}$, an optimal policy is greedy with respect to it:
$$
\pi^{\star}(s) = \arg\max_{a\in\mathcal{A}_{s}} Q^{\star}(s,a)\quad\forall s\in\mathcal{S}.
$$
The two algorithms below are two ways to solve the Bellman optimality equation for $V^{\star}$ (and hence $\pi^{\star}$).
___


## Value Iteration
Value iteration is a dynamic programming method that solves the Bellman optimality equation by repeated application. Define the __Bellman optimality operator__ $B$ acting on a value function $V:\mathcal{S}\rightarrow\mathbb{R}$:
$$
(BV)(s) = \max_{a\in\mathcal{A}_{s}}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}} P(s^{\prime}\mid s,a)\,V(s^{\prime})\right)\quad\forall s\in\mathcal{S}.
$$
Value iteration starts from an arbitrary $V_0$ and applies the operator repeatedly, $V_{k+1} = B V_{k}$:
$$
V_{k+1}(s) = \max_{a\in\mathcal{A}_{s}}\left(\underbrace{R(s,a)}_{\text{now}} + \gamma\underbrace{\sum_{s^{\prime}\in\mathcal{S}} P(s^{\prime}\mid s,a)\,V_{k}(s^{\prime})}_{\text{expected future value}}\right)\quad\forall s\in\mathcal{S}.
$$
At each sweep, the $\max$ operator makes a greedy choice using the current estimate $V_k$. Once the estimates stop changing, $V_k$ satisfies the Bellman optimality equation and equals $V^{\star}$.

### Algorithm
__Initialize:__ the MDP $\left(\mathcal{S},\mathcal{A},P,R,\gamma\right)$, a tolerance $\epsilon>0$, a maximum number of iterations $T_{\max}$, $V(s)\gets 0$ for all $s\in\mathcal{S}$, $\texttt{converged}\gets\texttt{false}$, and counter $k\gets 1$.

While $\texttt{converged}$ is $\texttt{false}$ __do:__
1. Set $\Delta\gets 0$.
2. For each $s\in\mathcal{S}$ __do:__
   - Backup: $V^{\prime}(s) \gets \max_{a\in\mathcal{A}_{s}}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}} P(s^{\prime}\mid s,a)\,V(s^{\prime})\right)$.
   - Track change: $\Delta\gets\max\left(\Delta,\;|V^{\prime}(s)-V(s)|\right)$.
3. Update $V(s)\gets V^{\prime}(s)$ for all $s\in\mathcal{S}$, and set $k\gets k+1$.
4. If $\Delta\le\epsilon$ (that is, $\lVert V^{\prime}-V\rVert_{\infty}\le\epsilon$) or $k>T_{\max}$, set $\texttt{converged}\gets\texttt{true}$.

The change measure $\Delta=\max_{s}|V^{\prime}(s)-V(s)|$ is the sup-norm $\lVert V^{\prime}-V\rVert_{\infty}$. On convergence, recover the policy with $\pi(s)=\arg\max_{a}\left(R(s,a)+\gamma\sum_{s^{\prime}}P(s^{\prime}\mid s,a)V(s^{\prime})\right)$.

### Convergence
The operator $B$ is a __contraction__ in the sup-norm with modulus $\gamma$:
$$
\lVert B V - B V^{\prime}\rVert_{\infty} \le \gamma\,\lVert V - V^{\prime}\rVert_{\infty}.
$$
Because $\gamma<1$, the Banach fixed-point theorem guarantees a unique fixed point $V^{\star}$ and that $\lim_{k\rightarrow\infty} V_k = V^{\star}$ from any starting $V_0$. The number of iterations to reach accuracy $\epsilon$ scales as
$$
T_{\max} \sim \left(\frac{1}{1-\gamma}\right)\ln\!\left(\frac{1}{\epsilon}\right).
$$

* **Long-term vs. short-term focus:** a $\gamma$ near $1$ values future rewards heavily, giving more far-sighted policies but slower convergence; a smaller $\gamma$ favors immediate rewards and converges faster.
* **Role of $\epsilon$:** a smaller tolerance $\epsilon$ yields a more accurate value function but requires more iterations.
* **Choosing $\gamma$:** set $\gamma$ from the planning horizon $H$ using $\gamma\approx 1-\tfrac{1}{H}$; for example $H=100$ gives $\gamma\approx 0.99$, and $H=10$ gives $\gamma\approx 0.90$.
___


## Policy Iteration
Policy iteration solves the same problem by alternating between evaluating a policy and improving it. Starting from an arbitrary policy $\pi_0$, it repeats two steps until the policy stops changing.

### Step 1: policy evaluation
Given a policy $\pi$, compute its value function $V^{\pi}$ by solving the Bellman expectation equation. Writing it in matrix form with the value vector $\mathbf{V}^{\pi}\in\mathbb{R}^{|\mathcal{S}|}$, the reward vector $\mathbf{R}^{\pi}$ with entries $R(s,\pi(s))$, and the $|\mathcal{S}|\times|\mathcal{S}|$ transition matrix $\mathbf{P}^{\pi}$ with entries $P(s^{\prime}\mid s,\pi(s))$:
$$
\mathbf{V}^{\pi} = \mathbf{R}^{\pi} + \gamma\,\mathbf{P}^{\pi}\mathbf{V}^{\pi}
\quad\Longrightarrow\quad
\mathbf{V}^{\pi} = \left(\mathbf{I} - \gamma\,\mathbf{P}^{\pi}\right)^{-1}\mathbf{R}^{\pi}.
$$
The matrix $\mathbf{I}-\gamma\mathbf{P}^{\pi}$ is invertible because $\gamma<1$ and $\mathbf{P}^{\pi}$ is a stochastic matrix. For large state spaces, $V^{\pi}$ is instead computed by iterating the Bellman expectation backup until the sup-norm change falls below a tolerance $\epsilon>0$.

### Step 2: policy improvement
Given $V^{\pi}$, define a new policy that is greedy with respect to it:
$$
\pi^{\prime}(s) = \arg\max_{a\in\mathcal{A}_{s}}\left(R(s,a) + \gamma\sum_{s^{\prime}\in\mathcal{S}} P(s^{\prime}\mid s,a)\,V^{\pi}(s^{\prime})\right)\quad\forall s\in\mathcal{S}.
$$
__Policy improvement theorem:__ the greedy policy is at least as good as the original, $V^{\pi^{\prime}}(s)\ge V^{\pi}(s)$ for all $s\in\mathcal{S}$, with equality only when $\pi$ is already optimal.

### Algorithm
__Initialize:__ the MDP $\left(\mathcal{S},\mathcal{A},P,R,\gamma\right)$ and an arbitrary policy $\pi$.

Repeat:
1. __Evaluate:__ solve $\mathbf{V}^{\pi}=\left(\mathbf{I}-\gamma\mathbf{P}^{\pi}\right)^{-1}\mathbf{R}^{\pi}$ for $V^{\pi}$.
2. __Improve:__ compute the greedy policy $\pi^{\prime}$ from $V^{\pi}$.
3. If $\pi^{\prime}=\pi$, stop and return $\pi^{\star}=\pi$; otherwise set $\pi\gets\pi^{\prime}$ and repeat.

For a finite MDP there are finitely many policies, and each iteration strictly improves the policy until none is better, so policy iteration converges to an optimal policy $\pi^{\star}$ in a finite number of steps.

### Value iteration vs. policy iteration
Both methods converge to the same optimal policy $\pi^{\star}$; they differ in the work per iteration.

| | Value iteration | Policy iteration |
| :--- | :--- | :--- |
| Per-iteration work | one Bellman optimality backup per state, cost $O(\lvert\mathcal{S}\rvert^{2}\lvert\mathcal{A}\rvert)$ | full policy evaluation (a linear solve, $O(\lvert\mathcal{S}\rvert^{3})$) plus one improvement step |
| Number of iterations | many cheap sweeps | few expensive iterations |
| Convergence | to within $\epsilon$ of $V^{\star}$ (contraction) | exact $\pi^{\star}$ in finitely many steps |

Value iteration takes many inexpensive steps; policy iteration takes a few expensive ones and usually converges in fewer iterations. Modified policy iteration, which evaluates a policy with a few backups instead of an exact solve, interpolates between the two.
___


## MDPs and Discrete Choice
Module 1 modeled a single discrete choice with a random utility model; this module models a sequence of choices with an MDP. The two connect directly through __dynamic discrete choice__ models, of which Rust's (1987) bus-engine replacement model is the canonical example.

In that model an agent decides each period whether to keep or replace a bus engine as a function of accumulated mileage (the state). The agent solves an MDP: choose the action in each state that maximizes the expected discounted return. This is the same optimal-stopping structure as the equipment-replacement, inventory, and portfolio-rebalancing problems used as graded examples in this course.

The link to Module 1 appears when each action's value carries a random shock, exactly the random utility model $U(s,a)=Q^{\star}(s,a)+\varepsilon_{a}$. If the shocks $\varepsilon_{a}$ are independent and identically distributed Gumbel (Type-I extreme value) across actions, the probability that the agent chooses action $a$ in state $s$ is the logit
$$
\Pr(a\mid s) = \frac{\exp\!\left(Q^{\star}(s,a)\right)}{\sum_{a^{\prime}\in\mathcal{A}_{s}}\exp\!\left(Q^{\star}(s,a^{\prime})\right)},
$$
the same multinomial logit form derived in Module 1, now with the action value $Q^{\star}(s,a)$ in the role of the deterministic utility. An MDP supplies the action values $Q^{\star}$, and the discrete choice model of Module 1 turns them into choice probabilities that can be estimated from observed decisions. The four modules are one toolkit: Markov models describe how states evolve, hidden Markov models infer states that are not observed, and MDPs choose actions that steer the evolution.
___


## Summary
This module extended Markov models into decision-making. An MDP adds actions and rewards to a Markov model, value functions measure how good states and actions are under a policy, and two dynamic programming algorithms compute the policy that maximizes expected discounted return.

> __Key takeaways:__
>
> 1. **MDPs and value functions:** A Markov decision process $(\mathcal{S},\mathcal{A},P,R,\gamma)$ adds actions and rewards to a Markov model. The state-value function $V^{\pi}(s)$ and action-value function $Q^{\pi}(s,a)$ measure the expected discounted return under a policy $\pi$ and satisfy the Bellman expectation equation; the optimal value functions satisfy the Bellman optimality equation, and the optimal policy is greedy with respect to $Q^{\star}$.
> 2. **Value iteration and policy iteration:** Value iteration applies the Bellman optimality operator repeatedly and converges to $V^{\star}$ because the operator is a sup-norm contraction with modulus $\gamma<1$. Policy iteration alternates exact policy evaluation with greedy policy improvement and reaches the optimal policy in finitely many steps by the policy improvement theorem. Both converge to $\pi^{\star}$ and trade many cheap iterations against a few expensive ones.
> 3. **Connection to discrete choice:** Dynamic discrete choice models such as Rust's (1987) bus-engine replacement are MDPs in which IID Gumbel shocks on the action values $Q^{\star}(s,a)$ produce the multinomial logit choice probabilities of Module 1, tying single-period discrete choice to sequential decision-making.

A Markov decision process turns a sequential decision problem into a value function that can be computed and a policy that can be acted on. With value iteration and policy iteration you can solve a formulated MDP and recover the optimal policy, the capstone tool of the course.

Exact dynamic programming becomes intractable for very large state spaces; two optional Advanced notebooks cover approximate planners, random rollout and Monte Carlo tree search (MCTS), for that setting.
___


### Additional Resources
This module draws on standard references in dynamic programming and reinforcement learning:
* Bellman, R. (1957). _Dynamic Programming_. Princeton University Press.
* Puterman, M. L. (1994). _Markov Decision Processes: Discrete Stochastic Dynamic Programming_. Wiley.
* Sutton, R. S., & Barto, A. G. (2018). _Reinforcement Learning: An Introduction_ (2nd ed.). MIT Press.
* Kochenderfer, M. J., Wheeler, T. A., & Wray, K. H. (2022). _Algorithms for Decision Making_. MIT Press.
* Rust, J. (1987). Optimal replacement of GMC bus engines: an empirical model of Harold Zurcher. _Econometrica_, 55(5), 999–1033.
